In [1]:
# Imports and Configurations
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import time

# Clustering and distance metrics
from dtaidistance import dtw
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.metrics import adjusted_rand_score

# Parallelization for OCP
import multiprocessing as mp

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

np.random.seed(42)

# Configuration
BENCHMARK        = '^GSPC'
RISK_FREE_RATE   = 0.045
START_DATE       = '2014-01-01'
END_DATE         = '2024-01-01'
ROLLING_WINDOW   = 52
MIN_HISTORY      = 400
K_VALUES         = [2,4,7]
N_CORES          = 14

# Three-period out-of-sample validation framework
TRAIN_START = '2014-01-01'
TRAIN_END   = '2017-12-31'
VAL_START   = '2018-01-01'
VAL_END     = '2020-12-31'
TEST_START  = '2021-01-01'
TEST_END    = '2023-12-31'

# Strategy Parameters
ZSCORE_WINDOW   = 52
VOL_WINDOW      = 26
VOL_THRESHOLD   = 1.5
HIGH_VOL_ENTRY  = 2.5
LOW_VOL_ENTRY   = 1.5
STANDARD_ENTRY  = 2.0
EXIT_THRESHOLD  = 0.5
STOP_LOSS       = 3.5

print('Notebook 6: DTW vs. OCP Clustering - S&P 500 Universe')
print(f'Period: {START_DATE} to {END_DATE}')
print(f'Risk-free rate: {RISK_FREE_RATE} (applied to both DTW and OCP Sharpe inputs)')
print(f'Parallelization: {N_CORES} cores for OCP')
print(f'\nThree-period framework (for later validation stage):')
print(f'    Train: {TRAIN_START} to {TRAIN_END}')
print(f'    Val:   {VAL_START} to {VAL_END}')
print(f'    Test:  {TEST_START} to {TEST_END}')
print(f'\nStrategy parameters (for later backtest stage):')
print(f'    Entry: ±{HIGH_VOL_ENTRY}σ (high vol) / ±{LOW_VOL_ENTRY}σ (low vol)')
print(f'    Exit:  ±{EXIT_THRESHOLD}σ')
print(f'    Stop:  ±{STOP_LOSS}σ')

Notebook 6: DTW vs. OCP Clustering - S&P 500 Universe
Period: 2014-01-01 to 2024-01-01
Risk-free rate: 0.045 (applied to both DTW and OCP Sharpe inputs)
Parallelization: 14 cores for OCP

Three-period framework (for later validation stage):
    Train: 2014-01-01 to 2017-12-31
    Val:   2018-01-01 to 2020-12-31
    Test:  2021-01-01 to 2023-12-31

Strategy parameters (for later backtest stage):
    Entry: ±2.5σ (high vol) / ±1.5σ (low vol)
    Exit:  ±0.5σ
    Stop:  ±3.5σ


In [2]:
# Retrieving S&P500 constituents as of January 1, 2014

# This source avoids survivorship bias
hist_url = (
    'https://raw.githubusercontent.com/fja05680/sp500/master/'
    'S%26P%20500%20Historical%20Components%20%26%20Changes.csv'
)

hist = pd.read_csv(hist_url)
hist['date'] = pd.to_datetime(hist['date'])

# Finding the row closest to our target date
target_date = pd.Timestamp('2014-01-01')
snapshot_row = hist[hist['date'] <= target_date].sort_values('date').iloc[-1]

print(f'Using snapshot date: {snapshot_row['date'].date()}')

sp500_tickers_2014 = sorted(snapshot_row['tickers'].split(','))

sp500_tickers_2014 = [t.strip().replace('.','-') for t in sp500_tickers_2014]

print(f'S&P 500 constituents as of {target_date.date()}: {len(sp500_tickers_2014)} stocks')
print(f'\nFirst 10 tickers: {sp500_tickers_2014[:10]}')

Using snapshot date: 2013-12-31
S&P 500 constituents as of 2014-01-01: 502 stocks

First 10 tickers: ['A', 'AABA', 'AAPL', 'ABBV', 'ABC', 'ABT', 'ACN', 'ADBE', 'ADI', 'ADM']


In [ ]:
# Downloading price date for the S&P 500 2014 universe
import yfinance as yf

print(f'Downloading price data for {len(sp500_tickers_2014)} stocks..')
print(f'Period: {START_DATE} to {END_DATE}, weekly (Wednesday), adjusted close\n')

raw = yf.download(
    sp500_tickers_2014,
    start=START_DATE,
    end=END_DATE,
    interval='1wk',
    auto_adjust=True,
    progress=True
)

price_data = raw['Close']
price_data = price_data.resample('W-WED').last()
price_data = price_data.dropna(how='all')

# Dropping any ticker without complete history across the full window 
before_drop = price_data.shape[1]
price_data  = price_data.ffill().dropna(axis=1)
after_drop  = price_data.shape[1]

print(f'\nPrice data shape: {price_data.shape}')
print(f'Date range: {price_data.index[0].date()} to {price_data.index[-1].date()}')
print(f'Tickers with complete history: {after_drop} (dropped {before_drop - after_drop} incomplete/delisted)')

dropped_tickers = sorted(set(sp500_tickers_2014) - set(price_data.columns))
print(f'\nDropped tickers ({len(dropped_tickers)}): {dropped_tickers}')

print("""
    Data source limitation note (for methodology section / paper writeup):

    yfinance (via Yahoo Finance) does not serve historical price data for
    tickers that have since delisted, regardless of how far in the past
    the requested date range is. This means out S&P 500 2014 universe is
    effectively: 'constituents as of Jan 2014 whose historical data 
    remains accessible via Yahoo Finance of the analysis date' (July 2026),
    not strictly 'constituents as of Jan 2014.' Companies that were acquired,
    taken private, or delisted at any point between 2014 and today are absent 
    even though they had complete price history during the actual 2014-2023 
    study window (verified via individual re-download diagnostic; confirmed 
    cases include WBA, acquired Sycamore Partners Aug 2025, and K/Kellanova,
    acquired by Mars Dec 2025).
""")

print("""
    This is a known constraint of free retail data vendors and differs 
    from the CRSP/WRDS standard in academic finance, which preserves
    full historical records for delisted securities. Disclosed here as
    a limitation rather than corrected, per project scope.
""")

Period: 2014-01-01 to 2024-01-01, weekly (Wednesday), adjusted close



$RHT: possibly delisted; no timezone found
$WYND: possibly delisted; no timezone found      ]  3 of 502 completed
[                       1%                       ]  7 of 502 completed$COG: possibly delisted; no timezone found
$FLIR: possibly delisted; no timezone found      ]  19 of 502 completed
$PXD: possibly delisted; no timezone found       ]  20 of 502 completed
$ALTR-201512: possibly delisted; no timezone found  21 of 502 completed
$RTN: possibly delisted; no timezone found
[**                     5%                       ]  23 of 502 completed$AGN: possibly delisted; no timezone found
[**                     5%                       ]  24 of 502 completed$STJ-201701: possibly delisted; no timezone found
[**                     5%                       ]  27 of 502 completed$PETM-201503: possibly delisted; no timezone found
[***                    6%                       ]  31 of 502 completed$K: possibly delisted; no timezone found
[***                    7%                   

In [ ]:
# Computing weekly log returns
log_returns = np.log(price_data / price_data.shift(1))

# Computing rolling sharpe ratio (risk-free-rate adjusted)
weekly_rf = RISK_FREE_RATE / 52

rolling_mean = log_returns.rolling(window=ROLLING_WINDOW).mean()
rolling_std  = log_returns.rolling(window=ROLLING_WINDOW).std()

sharpe_data = ((rolling_mean - weekly_rf) / rolling_std) * np.sqrt(52)
sharpe_data = sharpe_data.dropna(how='all')

print(f'Sharpe data shape: {sharpe_data.shape}')
print(f'Date range: {sharpe_data.index[0].date()} to {sharpe_data.index[-1].date()}')
print(f'\nSample (first 5 stocks, last 5 weeks):')
print(sharpe_data.iloc[-5:, :5])

In [ ]:
print(sharpe_data.index.day_name().unique())
print(f'\nTotal rows: {len(sharpe_data)}')
print(f'Duplicate dates: {sharpe_data.index.duplicated().sum()}')